# 🎯 Notebook 03 — Model Training

## AI Customer Sentiment Analysis Dashboard

---

**Objective:** Fine-tune DistilBERT on the preprocessed customer feedback dataset for 3-class sentiment classification.

**Model:** `distilbert-base-uncased-finetuned-sst-2-english` → fine-tuned with `num_labels=3`

**Metrics:** Accuracy, Macro F1, Per-class Precision/Recall

In [1]:
import os, warnings, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ PyTorch {torch.__version__} | Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

C:\Users\Acer\OneDrive\Desktop\Python Project\sentiment_dashboard\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyTorch 2.2.2+cpu | Device: cpu


## 1 · Configuration

In [2]:
# ── Training Config ───────────────────────────────────────────────
MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
OUTPUT_DIR = Path('../models/sentiment_model')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path('../data')
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 128
NUM_EPOCHS = 4          # Use 2 for quick demo, 4 for full training
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED = 42

LABEL2ID = {'NEGATIVE': 0, 'NEUTRAL': 1, 'POSITIVE': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print('Training Configuration:')
print(f'  Model:       {MODEL_NAME}')
print(f'  Epochs:      {NUM_EPOCHS}')
print(f'  Batch Size:  {BATCH_SIZE}')
print(f'  LR:          {LEARNING_RATE}')
print(f'  Max Length:  {MAX_LENGTH}')
print(f'  Output:      {OUTPUT_DIR}')

Training Configuration:
  Model:       distilbert-base-uncased-finetuned-sst-2-english
  Epochs:      4
  Batch Size:  16
  LR:          2e-05
  Max Length:  128
  Output:      ..\models\sentiment_model


## 2 · Load & Prepare Dataset

In [3]:
import re, html as html_module

def clean_text(text):
    if not isinstance(text, str) or not text.strip(): return ''
    text = html_module.unescape(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load splits
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df = pd.read_csv(DATA_DIR / 'val.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

# Clean and encode labels
for split_df in [train_df, val_df, test_df]:
    split_df['text_clean'] = split_df['text'].apply(clean_text)
    split_df['label_id'] = split_df['label'].map(LABEL2ID)

# Convert to HuggingFace Datasets
def df_to_dataset(df):
    return Dataset.from_pandas(df[['text_clean', 'label_id']].rename(
        columns={'text_clean': 'text', 'label_id': 'label'}
    ))

dataset = DatasetDict({
    'train': df_to_dataset(train_df),
    'validation': df_to_dataset(val_df),
    'test': df_to_dataset(test_df),
})

print(dataset)
print(f'\nLabel mapping: {LABEL2ID}')

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 3500
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 750
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 750
    })
})

Label mapping: {'NEGATIVE': 0, 'NEUTRAL': 1, 'POSITIVE': 2}


## 3 · Tokenization

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True,
                                 remove_columns=['text'])
tokenized_dataset.set_format('torch')

print('✅ Tokenization complete!')
print(f'   Features: {tokenized_dataset["train"].features}')
print(f'   Sample: {tokenized_dataset["train"][0]}')

Map: 100%|████████████████████████████████████| 750/750 [00:00<00:00, 20946.24 examples/s]

✅ Tokenization complete!
   Features: {'label': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}
   Sample: {'label': tensor(1), 'input_ids': tensor([ 101, 3492, 3115, 3325, 3452, 1012, 1996, 2946, 2003, 2074, 2157, 1012,
         102,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    

## 4 · Model Initialization

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,  # classifier head size changes from 2 to 3
)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model loaded: {MODEL_NAME}')
print(f'   Total parameters:     {total_params:>12,}')
print(f'   Trainable parameters: {trainable_params:>12,}')
print(f'   Frozen parameters:    {total_params - trainable_params:>12,}')

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased-finetuned-sst-2-english and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded: distilbert-base-uncased-finetuned-sst-2-english
   Total parameters:       66,955,779
   Trainable parameters:   66,955,779
   Frozen parameters:               0


## 5 · Define Metrics

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    return {
        'accuracy': acc,
        'f1': f1_macro,
        'f1_weighted': f1_weighted,
    }

print('✅ Metrics function defined.')

✅ Metrics function defined.


## 6 · Training Arguments & Trainer

In [9]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_strategy='steps',
    logging_steps=50,
    save_total_limit=2,
    seed=SEED,
    fp16=torch.cuda.is_available(),  # Mixed precision on GPU
    report_to='none',  # Set to 'wandb' if you have wandb configured
    dataloader_num_workers=0,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('✅ Trainer configured. Ready to train!')

✅ Trainer configured. Ready to train!


## 7 · Train the Model 🚀

In [ ]:
print('Starting training...')
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time
print(f'\n✅ Training complete in {elapsed/60:.1f} minutes')
print(f'   Final train loss: {train_result.training_loss:.4f}')

Starting training...


Epoch,Training Loss,Validation Loss


## 8 · Training Curves

In [ ]:
# Extract training history
log_history = trainer.state.log_history

# Training loss per step
train_steps = [x['step'] for x in log_history if 'loss' in x]
train_losses = [x['loss'] for x in log_history if 'loss' in x]

# Eval metrics per epoch
eval_entries = [x for x in log_history if 'eval_loss' in x]
eval_epochs = [x.get('epoch', i+1) for i, x in enumerate(eval_entries)]
eval_losses = [x['eval_loss'] for x in eval_entries]
eval_f1s = [x.get('eval_f1', 0) for x in eval_entries]
eval_accs = [x.get('eval_accuracy', 0) for x in eval_entries]

# Plot training curves
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=('Training Loss', 'Validation Loss & F1', 'Validation Accuracy'))

fig.add_trace(go.Scatter(x=train_steps, y=train_losses, mode='lines',
                         name='Train Loss', line=dict(color='#6c5ce7')), row=1, col=1)

fig.add_trace(go.Scatter(x=eval_epochs, y=eval_losses, mode='lines+markers',
                         name='Val Loss', line=dict(color='#FF6B6B')), row=1, col=2)
fig.add_trace(go.Scatter(x=eval_epochs, y=eval_f1s, mode='lines+markers',
                         name='Val F1', line=dict(color='#00C49A')), row=1, col=2)

fig.add_trace(go.Scatter(x=eval_epochs, y=eval_accs, mode='lines+markers',
                         name='Val Accuracy', line=dict(color='#FFB347')), row=1, col=3)

fig.update_layout(title='<b>Training Curves</b>', title_x=0.5,
                  template='plotly_dark', height=400)
fig.show()

## 9 · Evaluate on Validation Set

In [ ]:
val_results = trainer.evaluate()
print('Validation Results:')
for k, v in val_results.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

## 10 · Evaluate on Test Set

In [ ]:
test_results = trainer.evaluate(tokenized_dataset['test'])
print('Test Results:')
for k, v in test_results.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

# Detailed classification report
test_preds = trainer.predict(tokenized_dataset['test'])
pred_labels = np.argmax(test_preds.predictions, axis=-1)
true_labels = test_preds.label_ids

print('\nDetailed Classification Report:')
print(classification_report(true_labels, pred_labels, target_names=['NEGATIVE','NEUTRAL','POSITIVE']))

## 11 · Save Model & Tokenizer

In [ ]:
# Save the best model
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

# Save label mapping
label_config = {'label2id': LABEL2ID, 'id2label': ID2LABEL}
with open(OUTPUT_DIR / 'label_config.json', 'w') as f:
    json.dump(label_config, f, indent=2)

# Save training metrics
metrics = {
    'val_results': {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in val_results.items()},
    'test_results': {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in test_results.items()},
    'training_time_minutes': elapsed / 60,
    'num_epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
}
with open(REPORTS_DIR / 'training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'✅ Model saved to: {OUTPUT_DIR}')
print(f'✅ Metrics saved to: {REPORTS_DIR / "training_metrics.json"}')
print('\n✅ Notebook 03 Complete! Proceed to 04_evaluation.ipynb')